# Handling Sensitive data -- PII


## Column level redaction

In [0]:
CREATE OR REPLACE VIEW customer_vw AS
    SELECT
        customer_id,
        CASE
            WHEN is_account_group_member('admins_demo') THEN email
            ELSE 'REDACTED'
        END AS email,
        CASE
            WHEN is_account_group_member('admins_demo') THEN first_name
            ELSE 'REDACTED',
        END AS first_name,
        CASE
            WHEN is_account_group_member('admins_demo') THEN last_name
            ELSE 'REDACTED'
        END AS last_name,
        CASE
            WHEN is_account_group_member('admins_demo') THEN street
            ELSE 'REDACTED'
        END AS street,
        city,
        country
    FROM
        customer_details

## Row level access

In [0]:
CREATE OR REPLACE VIEW customer_fr_vw AS
    SELECT * FROM customer_vw
    WHERE
        CASE
            WHEN is_account_group_member('admins_demo') THEN TRUE
            ELSE country = 'France'
        END

## Masking logic in UDF


In [0]:
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURN CASE
    WHEN is_account_group_member('admins_demo') THEN email
    ELSE '***@***.***'
END;

In [0]:
-- apply the UDF for a TABLE LEVEL column masking
ALTER TABLE custoemr_details ALTER COLUMN email SET MASK mask_email;


## To dynamically filter rows on a table
To do this, start by defining filtering logic in UDF

If not a member of that group, then in this case, the query will only return rows for France with emails appearing as \*\*\*@\*\*\*.\*\*\*

If member, then the query returns all rows without filtering.

In [0]:
CREATE OR REPLACE FUNCTION geo_filter(country STRING)
RETURN
    IF(is_account_group_member('admins_demo'), true, country="France");


In [0]:
-- apply to table
ALTER TABLE customer_details SET ROW FILTER geo_filter ON (country);

## Remove all column masks and filters

In [0]:
ALTER TABLE customer_details ALTER COLUMN email DROP MASK;
ALTER TABLE customer_details DROP ROW FILTER;

# ABAC: Attribute-Based Access Control Policies
ABAC policies allow you to centrally define column mask and row fiilters based on table tags.